In [16]:
import os
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import pandas as pd
from tqdm import tqdm
import re
from pathlib import Path
from collections import defaultdict
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pyproj import Transformer
from scipy.stats import binned_statistic_2d
import rasterio
from rasterio.transform import from_bounds
from scipy.interpolate import griddata
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree
from dotenv import load_dotenv
from src import intertidal_topo
from src import swot_images_interface
import importlib
import xarray as xr
import pandas as pd
import holoviews as hv
import hvplot.pandas
import panel as pn
import tkinter as tk
from tkinter import messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from tkinterweb import HtmlFrame 
import webbrowser
from PyQt6.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel
#from PyQt6.QtWebEngineWidgets import QWebEngineView
import sys
import matplotlib.patches as patches

pn.extension()

importlib.reload(intertidal_topo)
importlib.reload(swot_images_interface)

/tmp/ipykernel_7594/26101375.py:40: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension()


<module 'src.swot_images_interface' from '/home/maytitan/Documents/intertidalSWOT-Tan/src/swot_images_interface.py'>

In [17]:
#-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-USER INPUTS-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.#
AOIName = "bassin_arcachon"              #Name of the AOI (Area Of Interest)
AOIType = 'geojson'                      #AOI file type : shp/geojson/KML/GPKG/CSV
StartDate = '2023-03-30'                    #SWOT Granule research Start Date
EndDate = '2023-09-30'                      #SWOT Granule research End Date
Reso = 20                                #Final resolution of the MNT
Method = 'Kmeans'                        #Method for extract intertidal topography : Kmeans/Mean5p100
WaterThreshold = 1                  #Maximum threshold of the prior_water_probability parameter
DistMaxInterpo = 2e3                 #Maximum search distance for points by the interpolators
OutputEPSG = 2154                   #EPSG of output files
DataWebsite = 'Earthaccess'           #Website where SWOT data are downloaded : Earthaccess/Hydroweb
MAJ_data = 'True'                      #Need to download SWOT data ? : True/False

#####Identification to https://search.earthdata.nasa.gov/
load_dotenv(dotenv_path="./.env")
if DataWebsite =='Earthaccess':
    if not os.path.exists(".env"):
        EarthDataUserName, EarthDataPassword = intertidal_topo.get_connexion_id_earthaccess()
    else:
        EarthDataUserName = os.getenv("EARTHDATA_USER_EARTHACCESS")
        EarthDataPassword = os.getenv("EARTHDATA_PASS_EARTHACCESS")
elif DataWebsite =='Hydroweb':
    if not os.path.exists(".env"):
        ApiKeyHydroweb = intertidal_topo.get_connexion_id_hydroweb()
    else:
        ApiKeyHydroweb = os.getenv("APIKEY_HYDROWEB")
#-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.-.#

AOIPath = "./AOI/"+AOIName+"."+AOIType.lower()
Project = os.path.splitext(os.path.basename(AOIPath))[0]

In [18]:
PathProject = "./tests/"+Project+StartDate+"_"+EndDate+Method
PathSWOT = PathProject+"/input/"
os.makedirs(PathSWOT, exist_ok=True)
PathOutput = PathProject+"/output/"
os.makedirs(PathOutput, exist_ok=True)
PathSWOTParquet = PathProject+"/Dataframe/"
os.makedirs(PathSWOTParquet, exist_ok=True)

In [19]:
drivers = {
    'shp': 'ESRI Shapefile',
    'geojson': 'GeoJSON',
    'kml': 'KML',
    'gpkg': 'GPKG'
}

if AOIType.lower() == 'csv':
    BoundingBox, AOIWGS84 = intertidal_topo.opening_AOI_csv(AOIPath)
    AOI_area = list(AOIWGS84.exterior.coords)
else:
    driver = drivers.get(AOIType.lower())
    if driver is None:
        raise ValueError(f"Format AOI inconnu : {AOIType}")
    BoundingBox, AOIWGS84 = intertidal_topo.opening_AOI(AOIPath, driver)

if MAJ_data == 'True':
    if DataWebsite =='Earthaccess':
        SWOTFiles = intertidal_topo.download_swot_pixelcloud_from_aoi_earthaccess(BoundingBox, StartDate, EndDate, PathSWOT, EarthDataUserName, EarthDataPassword)
    if DataWebsite =='Hydroweb':
        SWOTFiles = intertidal_topo.download_swot_pixelcloud_from_aoi_hydroweb(BoundingBox, StartDate, EndDate, PathSWOT, ApiKeyHydroweb, AOIName)
    SWOTFiles = intertidal_topo.keep_highest_version(SWOTFiles)
    if Method == "Kmeans": 
        SWOTFiles = swot_images_interface.afficher_fichiers(SWOTFiles, BoundingBox)
    SWOTData = intertidal_topo.filtering_data(SWOTFiles, AOIWGS84)
    SWOTData.to_parquet(PathSWOTParquet+AOIName+StartDate+".parquet", engine="pyarrow", index=False)
else:
    SWOTData = pd.read_parquet(PathSWOTParquet+AOIName+StartDate+".parquet", engine="pyarrow")
SWOTData

/home/maytitan/anaconda3/envs/earthaccess/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


10 granules trouvés dans l'AOI.


QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

10 granules with the most recent version.
Not enough points (0) in the bounding box pour tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/input/SWOT_L2_HR_PIXC_001_348_075R_20230802T151816_20230802T151827_PGC0_01.nc. Ignored.
Number of points to plot : 15567
Not enough points (0) in the bounding box pour tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/input/SWOT_L2_HR_PIXC_001_348_076L_20230802T151826_20230802T151837_PGC0_01.nc. Ignored.
Not enough points (0) in the bounding box pour tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/input/SWOT_L2_HR_PIXC_002_348_075R_20230823T120321_20230823T120332_PGC0_01.nc. Ignored.
Number of points to plot : 14382
Not enough points (0) in the bounding box pour tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/input/SWOT_L2_HR_PIXC_002_348_076L_20230823T120331_20230823T120342_PGC0_01.nc. Ignored.
Number of points to plot : 34612
Not enough points (0) in the bounding box pour tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/input/SWOT_L2_HR_PIXC_003_3

100%|██████████| 2/2 [00:11<00:00,  5.87s/it]


,points,height,sig0,coherent_power,classification,prior_water_prob,water_frac,bright_land_flag,false_detection_rate,inc,latitude,longitude,cycle,pass,tile,date_debut,date_fin,version,compteur,geometry
0,1852676,46.727680,31.898300,16522219.0,4.0,0.98,0.668109,0.0,0.000092,3.747736,44.598426,-1.214878,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59843)
1,1852677,46.595947,39.920563,20893112.0,4.0,0.98,0.595690,0.0,0.000101,3.748566,44.598447,-1.214994,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21499 44.59845)
2,1852678,46.518486,56.712433,29806268.0,4.0,0.98,0.810850,0.0,0.000111,3.749395,44.598469,-1.215119,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21512 44.59847)
3,1855065,46.393024,74.831261,39420024.0,4.0,0.98,1.221770,0.0,0.000097,3.748520,44.598628,-1.214881,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59863)
4,1855066,46.339378,140.447998,74544928.0,4.0,0.98,2.118137,0.0,0.000106,3.749349,44.598651,-1.215011,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21501 44.59865)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456184,4701686,46.170063,42.968849,23165218.0,4.0,0.99,0.949738,0.0,0.000113,1.469179,44.598850,-1.214714,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21471 44.59885)
456185,4701687,46.330708,20.465273,9418000.0,4.0,0.98,0.323528,0.0,0.000117,1.471328,44.598768,-1.215146,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21515 44.59877)
456186,4705390,46.342583,60.457745,30293048.0,4.0,0.98,1.070539,0.0,0.000102,1.469306,44.598642,-1.214746,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21475 44.59864)
456187,4705391,46.277199,42.923252,20782570.0,4.0,0.98,0.706462,0.0,0.000107,1.471455,44.598580,-1.215071,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21507 44.59858)


In [20]:
SWOTData = intertidal_topo.mask_water(SWOTData, WaterThreshold)

SWOTData

,points,height,sig0,coherent_power,classification,prior_water_prob,water_frac,bright_land_flag,false_detection_rate,inc,...,longitude,cycle,pass,tile,date_debut,date_fin,version,compteur,geometry,height_mask
0,1852676,46.727680,31.898300,16522219.0,4.0,0.98,0.668109,0.0,0.000092,3.747736,...,-1.214878,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59843),False
1,1852677,46.595947,39.920563,20893112.0,4.0,0.98,0.595690,0.0,0.000101,3.748566,...,-1.214994,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21499 44.59845),False
2,1852678,46.518486,56.712433,29806268.0,4.0,0.98,0.810850,0.0,0.000111,3.749395,...,-1.215119,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21512 44.59847),False
3,1855065,46.393024,74.831261,39420024.0,4.0,0.98,1.221770,0.0,0.000097,3.748520,...,-1.214881,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59863),False
4,1855066,46.339378,140.447998,74544928.0,4.0,0.98,2.118137,0.0,0.000106,3.749349,...,-1.215011,003,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21501 44.59865),False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456184,4701686,46.170063,42.968849,23165218.0,4.0,0.99,0.949738,0.0,0.000113,1.469179,...,-1.214714,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21471 44.59885),False
456185,4701687,46.330708,20.465273,9418000.0,4.0,0.98,0.323528,0.0,0.000117,1.471328,...,-1.215146,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21515 44.59877),False
456186,4705390,46.342583,60.457745,30293048.0,4.0,0.98,1.070539,0.0,0.000102,1.469306,...,-1.214746,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21475 44.59864),False
456187,4705391,46.277199,42.923252,20782570.0,4.0,0.98,0.706462,0.0,0.000107,1.471455,...,-1.215071,003,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21507 44.59858),False


In [21]:
SWOTData["X"], SWOTData["Y"] = intertidal_topo.LonLat2XY(SWOTData["longitude"], SWOTData["latitude"], 4326, OutputEPSG)

SWOTData

,points,height,sig0,coherent_power,classification,prior_water_prob,water_frac,bright_land_flag,false_detection_rate,inc,...,pass,tile,date_debut,date_fin,version,compteur,geometry,height_mask,X,Y
0,1852676,46.727680,31.898300,16522219.0,4.0,0.98,0.668109,0.0,0.000092,3.747736,...,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59843),False,365648.188225,6.397741e+06
1,1852677,46.595947,39.920563,20893112.0,4.0,0.98,0.595690,0.0,0.000101,3.748566,...,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21499 44.59845),False,365639.161780,6.397744e+06
2,1852678,46.518486,56.712433,29806268.0,4.0,0.98,0.810850,0.0,0.000111,3.749395,...,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21512 44.59847),False,365629.346289,6.397747e+06
3,1855065,46.393024,74.831261,39420024.0,4.0,0.98,1.221770,0.0,0.000097,3.748520,...,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21488 44.59863),False,365649.181878,6.397763e+06
4,1855066,46.339378,140.447998,74544928.0,4.0,0.98,2.118137,0.0,0.000106,3.749349,...,113,233L,20230904T234447,20230904T234458,PGC0,01,POINT (-1.21501 44.59865),False,365639.018697,6.397767e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456184,4701686,46.170063,42.968849,23165218.0,4.0,0.99,0.949738,0.0,0.000113,1.469179,...,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21471 44.59885),False,365663.738189,6.397787e+06
456185,4701687,46.330708,20.465273,9418000.0,4.0,0.98,0.323528,0.0,0.000117,1.471328,...,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21515 44.59877),False,365629.027549,6.397780e+06
456186,4705390,46.342583,60.457745,30293048.0,4.0,0.98,1.070539,0.0,0.000102,1.469306,...,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21475 44.59864),False,365659.944716,6.397764e+06
456187,4705391,46.277199,42.923252,20782570.0,4.0,0.98,0.706462,0.0,0.000107,1.471455,...,348,076R,20230913T084838,20230913T084849,PGC0,01,POINT (-1.21507 44.59858),False,365633.858679,6.397759e+06


In [22]:
GridSWOT, x_binning, y_binning, Width, Height, XGrid, YGrid = intertidal_topo.grid(BoundingBox, Reso, OutputEPSG)

GridSWOT

,X,Y
0,363117.253477,6.397855e+06
1,363137.253477,6.397855e+06
2,363157.253477,6.397855e+06
3,363177.253477,6.397855e+06
4,363197.253477,6.397855e+06
...,...,...
942247,383237.253477,6.416475e+06
942248,383257.253477,6.416475e+06
942249,383277.253477,6.416475e+06
942250,383297.253477,6.416475e+06


In [23]:
if Method == 'Mean5p100':
    GridSWOT = intertidal_topo.binning(SWOTData, GridSWOT, intertidal_topo.mean_lowest_5_percent, x_binning, y_binning, "height")

if Method == "Kmeans":
    SWOTData = intertidal_topo.Kmeans(SWOTData)
    SWOTData, IMaxSig0, IMinSig0 = intertidal_topo.association_classe_topo(SWOTData)
    SWOTData = intertidal_topo.apply_mask(SWOTData, "height", "height_mask", "Intertidal height")
    GridSWOT = intertidal_topo.gridding(SWOTData, GridSWOT, x_binning, y_binning, DistMaxInterpo)

GridSWOT['Longitude'], GridSWOT['Latitude'] = intertidal_topo.LonLat2XY(GridSWOT['X'], GridSWOT['Y'], OutputEPSG, 4326)

GridSWOT

IDW Interpolation: 100%|██████████| 95/95 [1:22:42<00:00, 52.23s/it]


,X,Y,height_avg,Intertidal height,Longitude,Latitude
0,363117.253477,6.397855e+06,NaN,NaN,-1.246800,44.598231
1,363137.253477,6.397855e+06,NaN,NaN,-1.246548,44.598241
2,363157.253477,6.397855e+06,NaN,NaN,-1.246296,44.598250
3,363177.253477,6.397855e+06,NaN,NaN,-1.246045,44.598260
4,363197.253477,6.397855e+06,NaN,NaN,-1.245793,44.598270
...,...,...,...,...,...,...
942247,383237.253477,6.416475e+06,NaN,NaN,-1.005508,44.775096
942248,383257.253477,6.416475e+06,NaN,NaN,-1.005255,44.775105
942249,383277.253477,6.416475e+06,NaN,NaN,-1.005003,44.775114
942250,383297.253477,6.416475e+06,NaN,NaN,-1.004750,44.775123


In [24]:
intertidal_topo.fig_recap(GridSWOT, SWOTData, PathOutput, Method+StartDate+"_"+EndDate+"recap.png", BoundingBox)
intertidal_topo.height_maps_per_tile(SWOTData, BoundingBox, output_dir=PathOutput+"height_per_tile")
if Method == "Kmeans":
    intertidal_topo.plot_classif_swot(SWOTData, PathOutput, Method+"classif.png", BoundingBox, IMaxSig0, IMinSig0)

In [25]:
intertidal_topo.export_raster(GridSWOT, XGrid, AOIName, Height, Width, PathOutput, Project, Method, OutputEPSG, Reso, StartDate, EndDate)

Export completed : ./tests/bassin_arcachon2023-03-30_2023-09-30Kmeans/output/Kmeansbassin_arcachon.tif
